# RAG Pipeline for Marketing Asset Generation
## INFO 290 Group 5 — Integrated Notebook
**Members:** Ryuichi Ikeda, Priya Charagondla, Shivani Parikh, Nseke Ngilbus

### Structure
| Section | Description |
|---------|-------------|
| 0 | Setup: dependencies, credentials, repo clone |
| 1 | Data: load TMDB corpus + genre mapping |
| 2 | RAG corpus: embed + FAISS index |
| 3 | Generator: Mistral-7B-Instruct-v0.2 (4-bit NF4) |
| 4 | Visual grounding: Gemini 2.5 Flash poster captioner |
| 4b | [Archive] Local VLM experiments (vit-gpt2, BLIP-2, Groq) |
| 5 | Offline poster caption batch processor |
| 6 | Core pipeline: retrieve → generate (V1 / V2) |
| 7 | LLM-as-Judge evaluation |
| 8 | Experiment: Idea 1 — Psychological Thriller |


## Section 0: Setup

In [1]:
# Cell 0-A: Install dependencies
!pip -q install "transformers>=4.41" "accelerate>=0.30" bitsandbytes \
                 sentence-transformers faiss-cpu pandas numpy requests tqdm \
                 google-generativeai openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 110.0 MB/s eta 0:00:00


In [2]:
# Cell 0-B: Imports & credentials
import os
import json
import re
import time
import base64
from io import BytesIO
from pprint import pprint

import numpy as np
import pandas as pd
import requests
import torch
import faiss
from tqdm import tqdm
from PIL import Image
from google.colab import userdata

# API credentials (stored in Colab Secrets)
TMDB_BEARER  = userdata.get("TMDB_BEARER")
HF_TOKEN     = userdata.get("HF_TOKEN")
GH_TOKEN     = userdata.get("GITHUB_TOKEN")
GEMINI_API_KEY  = userdata.get("GEMINI_API_KEY")
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("✅ Imports done")

✅ Imports done


In [3]:
# Cell 0-C: Clone repo
!git clone https://{GH_TOKEN}@github.com/ryuichi-github/ryuichi-github-info290-2026s-group5.git

Cloning into 'ryuichi-github-info290-2026s-group5'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 49 (delta 14), reused 37 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 2.55 MiB | 6.81 MiB/s, done.
Resolving deltas: 100% (14/14), done.


## Section 1: Data — Load TMDB Corpus

In [4]:
# Cell 1-A: Load dataset
REPO_ROOT = "ryuichi-github-info290-2026s-group5"
file_path = os.path.join(REPO_ROOT, "tmdb-fetch/tmdb_movies.csv")

if os.path.exists(file_path):
    movies_df = pd.read_csv(file_path)
    print(f"✅ Loaded {len(movies_df)} movies")
    print("Columns:", movies_df.columns.tolist())
else:
    raise FileNotFoundError(f"{file_path} not found. Check repo clone.")

✅ Loaded 4816 movies
Columns: ['id', 'title', 'tagline', 'overview', 'release_date', 'genre_ids', 'genre_names', 'vote_average', 'vote_count', 'popularity', 'poster_path', 'backdrop_path', 'runtime', 'original_language', 'status', 'revenue', 'budget', 'belongs_to_collection']


In [5]:
# Cell 1-B: TMDB API helper + genre mapping
def tmdb_get(url, params=None):
    headers = {
        "Authorization": f"Bearer {TMDB_BEARER}",
        "Content-Type": "application/json;charset=utf-8"
    }
    r = requests.get(url, headers=headers, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

# Verify API connection
tmdb_get("https://api.themoviedb.org/3/authentication")

# Genre id <-> name mapping
genre_data = tmdb_get(
    "https://api.themoviedb.org/3/genre/movie/list",
    params={"language": "en-US"}
)
GENRE_ID_TO_NAME = {g["id"]: g["name"] for g in genre_data.get("genres", [])}
GENRE_NAME_TO_ID = {v.lower(): k for k, v in GENRE_ID_TO_NAME.items()}

print(f"✅ Genres loaded: {len(GENRE_ID_TO_NAME)}")

✅ Genres loaded: 19


In [6]:
# Cell 1-C: Dataset subgroup analysis (genre distribution)
# Required for final report per professor feedback

# Parse genre_ids from string representation
corpus_preview = movies_df[
    movies_df["tagline"].notna() & (movies_df["tagline"].str.strip() != "")
].copy()

genre_counts = {}
for gids_str in corpus_preview["genre_ids"].dropna():
    try:
        for gid in eval(gids_str):
            name = GENRE_ID_TO_NAME.get(gid, f"id:{gid}")
            genre_counts[name] = genre_counts.get(name, 0) + 1
    except Exception:
        pass

genre_series = pd.Series(genre_counts).sort_values(ascending=False)
print("Genre distribution in corpus (movies with taglines):")
print(genre_series.to_string())
print(f"\nTotal corpus size: {len(corpus_preview)}")
print("\nNote: one movie can belong to multiple genres.")

Genre distribution in corpus (movies with taglines):
Drama              1845
Action             1414
Thriller           1366
Comedy             1221
Adventure           939
Crime               751
Science Fiction     641
Horror              632
Romance             632
Fantasy             558
Family              506
Animation           389
Mystery             381
History             230
War                 127
Music               102
Western              70
TV Movie             32
Documentary          19

Total corpus size: 4312

Note: one movie can belong to multiple genres.


## Section 2: RAG Corpus — Embed + FAISS Index

In [7]:
# Cell 2: Build RAG corpus
# Corpus  : movies with non-empty taglines (4,312 of 4,816)
# Embedding: title + tagline + overview → Sentence-BERT (all-MiniLM-L6-v2, 384-dim)
# Index   : FAISS IndexFlatIP (inner product on L2-normalised vectors = cosine similarity)

from sentence_transformers import SentenceTransformer

corpus_df = movies_df[
    movies_df["tagline"].notna() & (movies_df["tagline"].str.strip() != "")
].copy().reset_index(drop=True)

print(f"Total movies : {len(movies_df)}")
print(f"Corpus size  : {len(corpus_df)} ({len(corpus_df)/len(movies_df)*100:.1f}%)")
print(f"Excluded     : {len(movies_df) - len(corpus_df)} (missing tagline)")

embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

corpus_texts = (
    corpus_df["title"] + " " + corpus_df["tagline"] + " " + corpus_df["overview"]
).tolist()

embeddings = embed_model.encode(
    corpus_texts,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

dim = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(embeddings)

print(f"✅ Embeddings : {embeddings.shape}")
print(f"✅ FAISS index: {faiss_index.ntotal} vectors")

Total movies : 4816
Corpus size  : 4312 (89.5%)
Excluded     : 504 (missing tagline)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/135 [00:00<?, ?it/s]

✅ Embeddings : (4312, 384)
✅ FAISS index: 4312 vectors


## Section 3: Generator — Mistral-7B-Instruct-v0.2 (4-bit NF4)

In [8]:
# Cell 3: Load LLM
# Model  : Mistral-7B-Instruct-v0.2
# Quant  : 4-bit NF4 (bitsandbytes) — fits in T4 16GB VRAM

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    device_map="auto",
    quantization_config=bnb_config,
)

print("✅ Model loaded on:", model.device)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

✅ Model loaded on: cuda:0


## Section 4: Visual Grounding — Gemini 2.5 Flash (Adopted)

**Decision:** Gemini 2.5 Flash via API is the adopted captioner.  
All other VLM experiments (vit-gpt2, BLIP-2, Groq Llama 4 Scout) are archived in Section 4b.

**6-axis caption structure:**
1. Composition (subject position, angle, negative space)
2. Lighting & shadows (source direction, silhouette)
3. Color palette (dominant tone, accent colors)
4. Emotional tone (unease, isolation, tension...)
5. Typography relationship (text position, image interaction)
6. Iconic visual symbol (one defining object)


In [9]:
# Cell 4-A: Gemini 2.5 Flash setup
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)
gemini = genai.GenerativeModel("gemini-2.5-flash")

TMDB_IMAGE_BASE = "https://image.tmdb.org/t/p/w780"  # w780 keeps 2:3 ratio without cropping

POSTER_PROMPT = """Analyze this movie poster and write a detailed description in flowing prose.
Do not use bullet points, numbers, or lists.
Do not invent, assume, or hallucinate any details not visible in the image.

Your description must cover:
(1) composition and positioning of subjects,
(2) direction and quality of lighting and shadows,
(3) overall color palette including dominant and accent colors,
(4) emotional mood and atmosphere,
(5) typography including text position and interaction with subjects,
(6) one key iconic visual symbol that defines the poster.

Be specific, vivid, and literal. This description will be used by a designer to
recreate the visual style for a new movie poster."""

print("✅ Gemini 2.5 Flash ready")

✅ Gemini 2.5 Flash ready


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [10]:
# Cell 4-B: describe_poster_gemini() — single poster captioner
def describe_poster_gemini(poster_path: str) -> str:
    """
    Fetch a TMDB poster and return a 6-axis prose caption via Gemini 2.5 Flash.

    Parameters
    ----------
    poster_path : str
        TMDB poster_path field (e.g. '/dldIX0q5jewe8rSyCh8d5I1RYx3.jpg')

    Returns
    -------
    str
        Prose caption, or empty string on failure.
    """
    if not poster_path or pd.isna(poster_path):
        return ""
    try:
        url = TMDB_IMAGE_BASE + poster_path
        r = requests.get(url, timeout=15, stream=True)
        img_bytes = b"".join(chunk for chunk in r.iter_content(chunk_size=4096) if chunk)
        img = Image.open(BytesIO(img_bytes)).convert("RGB")
        response = gemini.generate_content([img, POSTER_PROMPT])
        return response.text.strip()
    except Exception as e:
        print(f"Poster description failed for {poster_path}: {e}")
        return ""

print("✅ describe_poster_gemini ready")

✅ describe_poster_gemini ready


In [11]:
# Cell 4-C: Single-poster test (Take Shelter)
test_path = "/dldIX0q5jewe8rSyCh8d5I1RYx3.jpg"
caption = describe_poster_gemini(test_path)
print(f"Caption:\n{caption}")

Caption:
The poster immediately conveys a profound sense of foreboding and existential dread, dominated by an overwhelming, turbulent sky that looms above a small, anxious family. In the foreground, positioned slightly to the left and occupying a significant portion of the lower frame, stands the central male figure, his gaze directed intently upwards and slightly to the right, his brow furrowed with deep concern. Behind him and to his right, partially obscured, is a woman with striking red hair, her anxious expression mirroring his as she, too, stares skyward. Further to her right and slightly lower, a young girl’s face is partially visible, also looking up, completing the family’s shared apprehension.

The lighting throughout the poster is dim and diffused, characteristic of an approaching storm, with subtle shadows defining the contours of the man’s face and the folds of his shirt, suggesting an indirect light source from above and slightly to his left. The overall color palette is 

## Section 4b: [Archive] Local VLM Experiments

These cells document the VLM comparison that led to adopting Gemini 2.5 Flash.  
**Do not run in production** — these require GPU memory alongside Mistral-7B.

| Model | Verdict | Reason |
|-------|---------|--------|
| vit-gpt2 (200MB, CPU) | Rejected | No prompt support, object misrecognition |
| BLIP-2 zero-shot (3.9B, CPU) | Rejected | Reads poster text only |
| BLIP-2 Q&A prompt (3.9B, CPU) | Insufficient | Context collapse ("dark and moody" for all) |
| Groq Llama 4 Scout (API) | Rejected | Hallucinated fictional titles in generation |
| **Gemini 2.5 Flash (API)** | **Adopted** | Accurate 6-axis prose, no hallucination |


In [ ]:
# [ARCHIVE] vit-gpt2 experiment — do not run in production
# from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer as VitTokenizer
# vit_model = VisionEncoderDecoderModel.from_pretrained("nlpconnect/vit-gpt2-image-captioning").to("cpu")
# vit_processor = ViTImageProcessor.from_pretrained("nlpconnect/vit-gpt2-image-captioning")
# vit_tokenizer = VitTokenizer.from_pretrained("nlpconnect/vit-gpt2-image-captioning")
# Result on Take Shelter: "man standing in front of large group of birds" ← rejected

In [ ]:
# [ARCHIVE] BLIP-2 experiment — do not run in production
# from transformers import Blip2Processor, Blip2ForConditionalGeneration
# blip_processor = Blip2Processor.from_pretrained("Salesforce/blip2-flan-t5-xl")
# blip_model = Blip2ForConditionalGeneration.from_pretrained(
#     "Salesforce/blip2-flan-t5-xl", torch_dtype=torch.float32
# ).to("cpu")
# Zero-shot result: "the poster is dark and moody" ← context collapse, rejected
# Q&A result: "a man and woman standing in front of a stormy sky" ← too shallow

In [ ]:
# [ARCHIVE] Groq Llama 4 Scout — do not use for generation pipeline
# Rejected because it hallucinated a fictional title "THE ISOLATED" in poster_art_direction output.
# The poster captioning quality itself was acceptable but the downstream contamination was disqualifying.
#
# from groq import Groq
# GROQ_API_KEY = userdata.get("GROQ_API_KEY")
# groq_client = Groq(api_key=GROQ_API_KEY)

## Section 5: Offline Poster Caption Batch Processor

Pre-process all 4,312 posters with Gemini 2.5 Flash and save captions to CSV.  
At inference time, captions are looked up by film ID — no runtime API calls needed.

**Rationale:** Retrieval axis is narrative/marketing similarity (textual), not visual similarity.  
Poster captions are supplementary context for generation, not a search dimension.

**Cost estimate:** ~$5 at Gemini 2.5 Flash paid tier for 4,312 images.


In [12]:
# Cell 5-A: Offline batch captioner
# Saves captions to poster_captions.csv with checkpoints every 1,000 images.

CAPTION_OUTPUT = os.path.join(REPO_ROOT, "tmdb-fetch/poster_captions.csv")
CHECKPOINT_INTERVAL = 1000

def batch_caption_posters(
    df: pd.DataFrame,
    output_path: str,
    checkpoint_interval: int = 1000,
    sleep_between: float = 0.3,
) -> pd.DataFrame:
    """
    Generate Gemini 2.5 Flash captions for all posters in df.

    Resumes from existing output_path if present (skips already-captioned rows).
    Saves a checkpoint every `checkpoint_interval` new captions.

    Parameters
    ----------
    df               : DataFrame with columns [id, title, poster_path]
    output_path      : path to save/resume the caption CSV
    checkpoint_interval : save frequency
    sleep_between    : seconds between API calls (rate-limit buffer)

    Returns
    -------
    DataFrame with columns [id, title, poster_path, poster_caption]
    """
    # Resume from existing file
    if os.path.exists(output_path):
        done = pd.read_csv(output_path, usecols=["id"])
        done_ids = set(done["id"].tolist())
        print(f"Resuming: {len(done_ids)} already done, {len(df) - len(done_ids)} remaining.")
    else:
        done_ids = set()
        print(f"Starting fresh: {len(df)} posters to process.")

    results = []
    new_count = 0

    for _, row in tqdm(df.iterrows(), total=len(df)):
        if row["id"] in done_ids:
            continue

        caption = describe_poster_gemini(row.get("poster_path", None))
        results.append({
            "id": row["id"],
            "title": row["title"],
            "poster_path": row.get("poster_path", ""),
            "poster_caption": caption,
        })
        new_count += 1
        time.sleep(sleep_between)

        # Checkpoint
        if new_count % checkpoint_interval == 0:
            chunk = pd.DataFrame(results)
            chunk.to_csv(
                output_path,
                mode="a",
                header=not os.path.exists(output_path),
                index=False
            )
            results = []
            print(f"  Checkpoint saved at {new_count} new captions.")

    # Final flush
    if results:
        chunk = pd.DataFrame(results)
        chunk.to_csv(
            output_path,
            mode="a",
            header=not os.path.exists(output_path),
            index=False
        )

    # Return full merged result
    return pd.read_csv(output_path)

print("✅ batch_caption_posters ready")

✅ batch_caption_posters ready


In [13]:
# Cell 5-B: Run batch captioning
# WARNING: costs ~$5 in Gemini API credits. Run once and commit the CSV.

# caption_df = batch_caption_posters(
#     df=corpus_df[["id", "title", "poster_path"]],
#     output_path=CAPTION_OUTPUT,
# )
# print(f"✅ Captioning complete. Total rows: {len(caption_df)}")

# To load pre-generated captions:
if os.path.exists(CAPTION_OUTPUT):
    caption_df = pd.read_csv(CAPTION_OUTPUT)
    corpus_df = corpus_df.merge(
        caption_df[["id", "poster_caption"]], on="id", how="left"
    )
    print(f"✅ Captions loaded. Coverage: {corpus_df['poster_caption'].notna().sum()}/{len(corpus_df)}")
else:
    print("poster_captions.csv not found. Run batch captioning first.")

poster_captions.csv not found. Run batch captioning first.


## Section 6: Core Pipeline — Retrieval, Generation, V1/V2

**Fixed variables across all experiments:**
- Retrieval corpus: 4,312 movies
- k = 5, genre filter applied
- Embedding model: all-MiniLM-L6-v2
- Generator: Mistral-7B-Instruct-v0.2 (4-bit NF4)
- Input format: structured dict (core_premise / thematic_core / negative_constraints)


In [14]:
# Cell 6-A: Genre filter helper
def _normalize_genre_filter(genre_filter):
    """
    Accepts None, list of genre name strings, or list of genre int IDs.
    Returns list of int IDs or None.
    """
    if genre_filter is None:
        return None
    if isinstance(genre_filter, (list, tuple)) and len(genre_filter) > 0:
        if isinstance(genre_filter[0], str):
            ids = [GENRE_NAME_TO_ID[name.lower()] for name in genre_filter
                   if name.lower() in GENRE_NAME_TO_ID]
            return ids if ids else []
        if isinstance(genre_filter[0], int):
            return list(genre_filter)
    raise ValueError("genre_filter must be None, list of genre name strings, or list of genre int IDs.")

In [15]:
# Cell 6-B: Retrieval
def retrieve(query: str, k: int = 5, genre_filter=None) -> pd.DataFrame:
    """
    Retrieve top-k similar movies from the corpus.

    Parameters
    ----------
    query        : free-text query (typically core_premise)
    k            : number of results to return
    genre_filter : list of genre names or IDs; None = no filter

    Returns
    -------
    DataFrame with top-k rows from corpus_df plus a 'score' column.
    Falls back to unfiltered search if no genre-matching candidates exist.
    """
    q = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    gf_ids = _normalize_genre_filter(genre_filter)

    if gf_ids is None:
        scores, idx = faiss_index.search(q, k)
        out = corpus_df.iloc[idx[0]].copy()
        out["score"] = scores[0]
        return out.reset_index(drop=True)

    gf_ids = set(gf_ids)
    mask = corpus_df["genre_ids"].apply(
        lambda gids: any(gid in gf_ids for gid in eval(gids))
    ).to_numpy()
    candidate_idx = np.where(mask)[0]

    if candidate_idx.size == 0:
        scores, idx = faiss_index.search(q, k)
        out = corpus_df.iloc[idx[0]].copy()
        out["score"] = scores[0]
        out["note"] = "No genre-filtered candidates; fell back to unfiltered."
        return out.reset_index(drop=True)

    cand_emb = embeddings[candidate_idx]
    scores = cand_emb @ q[0]
    topk_local = np.argsort(-scores)[:k]
    topk_idx = candidate_idx[topk_local]
    out = corpus_df.iloc[topk_idx].copy()
    out["score"] = scores[topk_local]
    return out.reset_index(drop=True)

In [16]:
# Cell 6-C: Reference text builder
def refs_to_text(df: pd.DataFrame, n: int = 5, use_visual: bool = True) -> str:
    """
    Format retrieved movies as a reference string for the prompt.

    Includes tagline + overview (truncated to 200 chars) as style reference.
    Optionally appends pre-computed poster caption if use_visual=True and
    poster_caption column is present.

    Parameters
    ----------
    df         : top-k retrieval result from retrieve()
    n          : number of references to include
    use_visual : whether to include poster_caption (V2 only)
    """
    lines = []
    for _, r in df.head(n).iterrows():
        if pd.notna(r["tagline"]) and r["tagline"].strip():
            ov = (r["overview"] or "")[:200].replace("\n", " ")
            line = f'- {r["title"]}: tagline: "{r["tagline"]}" | overview: "{ov}"'
            if use_visual and "poster_caption" in r and pd.notna(r.get("poster_caption")):
                caption = str(r["poster_caption"]).strip()
                if caption:
                    line += f' | visual style: "{caption[:300]}"'
            lines.append(line)
    return "\n".join(lines)

In [17]:
# Cell 6-D: Text generation
def generate_text(prompt: str, max_new_tokens: int = 320, temperature: float = 0.6) -> str:
    """
    Generate text with Mistral-7B using greedy-ish sampling.

    Parameters
    ----------
    prompt         : full formatted prompt string
    max_new_tokens : generation budget (320 is sufficient for JSON output)
    temperature    : sampling temperature for do_sample generation
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.10,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

In [18]:
# Cell 6-E: JSON extraction
REQUIRED_KEYS = {"overview", "tagline", "poster_art_direction"}

def extract_best_json(text: str) -> dict:
    """
    Extract the best-matching JSON object from raw model output.

    Scans the full text for JSON objects and scores each candidate by
    presence of required keys and plausible field lengths.
    Raises ValueError if no JSON object is found.
    """
    decoder = json.JSONDecoder()
    objs = []
    i = 0
    while True:
        start = text.find("{", i)
        if start == -1:
            break
        try:
            obj, end = decoder.raw_decode(text[start:])
            objs.append(obj)
            i = start + end
        except json.JSONDecodeError:
            i = start + 1

    if not objs:
        raise ValueError("No JSON object found in model output.")

    def score(obj):
        if not isinstance(obj, dict):
            return -10
        if not REQUIRED_KEYS.issubset(set(obj.keys())):
            return -5
        s = 0
        tagline  = obj.get("tagline", "")
        poster   = obj.get("poster_art_direction", "")
        overview = obj.get("overview", "")
        if isinstance(tagline, str) and len(tagline.strip()) > 0:
            s += 2
            if len(tagline.split()) <= 14:
                s += 1
        if isinstance(poster, str) and len(poster.strip()) > 20:
            s += 2
        if isinstance(overview, str) and len(overview.strip()) > 40:
            s += 2
        return s

    return max(objs, key=score)

In [19]:
# Cell 6-F: Prompt builder
def build_prompt(movie_input: dict, refs: str) -> str:
    """
    Build the generation prompt from structured input and retrieved references.

    movie_input keys: core_premise, thematic_core, negative_constraints
    refs            : formatted reference string from refs_to_text()
    """
    premise   = movie_input.get("core_premise", "")
    theme     = movie_input.get("thematic_core", "")
    negatives = movie_input.get("negative_constraints", "")

    return f"""You are a movie marketing generator.

Generate marketing assets for the movie described below.

OUTPUT: ONE JSON object with EXACTLY these keys:
- overview (<= 80 words, compelling promo synopsis)
- tagline (<= 12 words)
- poster_art_direction (<= 60 words)

JSON FORMAT:
{{"overview": "", "tagline": "", "poster_art_direction": ""}}

MOVIE DETAILS:
Core Premise: {premise}
Thematic Core: {theme}
IMPORTANT - Do NOT do the following: {negatives}

REFERENCES from similar successful movies (use for tone and syntax ONLY; do NOT copy plot):
{refs}

CONSTRAINTS:
- Output ONLY the JSON object (no markdown, no backticks, no commentary).
- tagline must be original and specific to this movie, not generic.
- overview must be written as compelling promo copy, not a plot summary.
- End immediately after the final '}}'.

Now output the JSON:
"""

print("✅ build_prompt ready")

✅ build_prompt ready


In [20]:
# Cell 6-G: V1 / V2 runner
def run_v1_v2(movie_input: dict, k: int = 5, genre_filter=None, temperature: float = 0.6):
    """
    Run both V1 (zero-shot) and V2 (RAG + visual grounding) for a given input.

    V1: no retrieval, prompt receives refs="(none)"
    V2: retrieves top-k, includes poster_caption if available (use_visual=True)

    Returns
    -------
    (v1_output, v2_output, topk_df, refs_string)
    """
    query = movie_input["core_premise"]

    # V1: zero-shot baseline
    p1 = build_prompt(movie_input, refs="(none)")
    t1 = generate_text(p1, temperature=temperature)
    j1 = extract_best_json(t1)

    # V2: RAG + visual grounding
    topk = retrieve(query, k=k, genre_filter=genre_filter)
    refs = refs_to_text(topk, n=k, use_visual=True)
    p2 = build_prompt(movie_input, refs=refs)
    t2 = generate_text(p2, temperature=temperature)
    j2 = extract_best_json(t2)

    return j1, j2, topk, refs

print("✅ run_v1_v2 ready")

✅ run_v1_v2 ready


## Section 7: LLM-as-Judge Evaluation

**Design decisions:**
- Judge model: gpt-4o (different from generator Mistral-7B — avoids self-enhancement bias)
- Evaluation axes: 5 dimensions matching observed failure modes during development
- Scoring: 1–5 integer per dimension + one-sentence reason (for interpretability)
- Batch mode: `evaluate_batch()` for ~200 QA pairs; `summarize_batch()` for aggregated stats

**Reference:** MT-Bench / Chatbot Arena (Zheng et al., 2023) for LLM-as-Judge methodology.


In [21]:
# Cell 7-A: Rubric definition
from openai import OpenAI

JUDGE_MODEL = "gpt-4o"

RUBRIC = {
    "narrative_fidelity": {
        "description": "Output follows intended premise without adding unintended plot elements",
        "5": "No negative_constraints violated. All content anchored strictly to core_premise and thematic_core.",
        "3": "Minor trope drift or one incidental element not in the premise; no invented characters or major additions.",
        "1": "Invented characters, disappearances, villains, or plot elements explicitly excluded by negative_constraints.",
    },
    "genre_alignment": {
        "description": "Output matches the intended genre and tone",
        "5": "Vocabulary, sentence rhythm, and emotional register fully match the target genre.",
        "3": "Mostly aligned but one phrase or image feels like it belongs to a different genre.",
        "1": "Clearly wrong tone — e.g., action-thriller language for a slow psychological piece.",
    },
    "visual_specificity": {
        "description": "Poster description includes concrete, actionable visual elements",
        "5": "Specific composition, lighting source, colour palette, and at least one symbolic object.",
        "3": "Some concrete details but partially falls back on generic descriptors.",
        "1": "Only generic descriptors such as 'dark and moody' with no actionable composition detail.",
    },
    "creative_specificity": {
        "description": "Output contains distinctive, non-generic phrasing",
        "5": "A copywriter or creative director would keep this line without revision.",
        "3": "Adequate and on-brief, but forgettable — could have been written by any baseline LLM.",
        "1": "Cliché, fill-in-the-blank template, or a phrase appearing verbatim in retrieved references.",
    },
    "output_format_validity": {
        "description": "Output follows required structure and word limits",
        "5": "All three fields present. tagline ≤ 12 words, overview ≤ 80 words, poster_art_direction ≤ 60 words.",
        "3": "All fields present but one field slightly exceeds the word limit (≤ 20% over).",
        "1": "A required field is missing, or one field is more than 20% over the word limit.",
    },
}

DIMENSIONS = list(RUBRIC.keys())
print("✅ Rubric defined")

✅ Rubric defined


In [22]:
# Cell 7-B: Prompt builder for judge
def _build_judge_prompt(movie_input: dict, generated_output: dict) -> str:
    rubric_text = ""
    for dim, content in RUBRIC.items():
        rubric_text += f"\n{dim}:\n"
        rubric_text += f"  5 = {content['5']}\n"
        rubric_text += f"  3 = {content['3']}\n"
        rubric_text += f"  1 = {content['1']}\n"

    return f"""You are an expert film marketing evaluator.
Score the following generated marketing assets on 5 dimensions, each on a 1–5 integer scale.

[INPUT CONCEPT]
core_premise: {movie_input.get('core_premise', '')}
thematic_core: {movie_input.get('thematic_core', '')}
negative_constraints: {movie_input.get('negative_constraints', '')}

[GENERATED OUTPUTS]
tagline: {generated_output.get('tagline', '')}
overview: {generated_output.get('overview', '')}
poster_art_direction: {generated_output.get('poster_art_direction', '')}

[RUBRIC]{rubric_text}
Scoring rules:
- Score each dimension independently.
- Assign integer scores only: 1, 2, 3, 4, or 5.
- Write one concise sentence as reason for each score.
- Do NOT copy phrasing from the generated output in your reasons.

Return ONLY valid JSON. No markdown, no preamble. Schema:
{{
  "scores": {{
    "narrative_fidelity":      {{"score": <int>, "reason": "<str>"}},
    "genre_alignment":         {{"score": <int>, "reason": "<str>"}},
    "visual_specificity":      {{"score": <int>, "reason": "<str>"}},
    "creative_specificity":    {{"score": <int>, "reason": "<str>"}},
    "output_format_validity":  {{"score": <int>, "reason": "<str>"}}
  }},
  "average": <float>
}}"""

print("✅ _build_judge_prompt ready")

✅ _build_judge_prompt ready


In [23]:
# Cell 7-C: llm_as_judge() — single evaluation
def llm_as_judge(
    movie_input: dict,
    generated_output: dict,
    client: OpenAI = None,
    max_retries: int = 3,
    retry_delay: float = 2.0,
    verbose: bool = True,
) -> dict:
    """
    Score a single generated output on 5 dimensions using GPT-4o as judge.

    Parameters
    ----------
    movie_input       : structured input dict (core_premise, thematic_core, negative_constraints)
    generated_output  : generated dict (tagline, overview, poster_art_direction)
    client            : OpenAI instance; created from env var if None
    max_retries       : JSON parse retry attempts
    retry_delay       : seconds between retries
    verbose           : print score summary if True

    Returns
    -------
    dict with keys: scores (per dimension), average (float), error (str|None)
    """
    if client is None:
        client = OpenAI()

    prompt = _build_judge_prompt(movie_input, generated_output)

    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=JUDGE_MODEL,
                max_tokens=1000,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": (
                        "You are a strict film marketing evaluator. "
                        "Return only valid JSON matching the requested schema. "
                        "No markdown, no preamble, no explanation outside the JSON."
                    )},
                    {"role": "user", "content": prompt},
                ],
            )
            raw_text = response.choices[0].message.content.strip()
            raw_text = re.sub(r"^```(?:json)?\s*", "", raw_text)
            raw_text = re.sub(r"\s*```$", "", raw_text)

            result = json.loads(raw_text)

            # Recompute average to guard against judge arithmetic errors
            scores = result.get("scores", {})
            vals = [v["score"] for v in scores.values() if isinstance(v.get("score"), (int, float))]
            result["average"] = round(sum(vals) / len(vals), 2) if vals else 0.0
            result["error"] = None

            if verbose:
                _print_judge_result(result)
            return result

        except (json.JSONDecodeError, KeyError) as e:
            if attempt < max_retries:
                if verbose:
                    print(f"[judge] attempt {attempt} failed ({e}), retrying in {retry_delay}s...")
                time.sleep(retry_delay)
            else:
                return {
                    "scores": {},
                    "average": 0.0,
                    "error": str(e),
                    "raw": raw_text if "raw_text" in dir() else "",
                }


def _print_judge_result(result: dict) -> None:
    print("\n" + "=" * 52)
    print(f"  average: {result['average']:.2f} / 5.00")
    print("=" * 52)
    for dim in DIMENSIONS:
        entry = result["scores"].get(dim, {})
        score  = entry.get("score", "?")
        reason = entry.get("reason", "")
        bar = "█" * score + "░" * (5 - score) if isinstance(score, int) else ""
        print(f"  {dim:<28} {bar}  {score}/5")
        print(f"    → {reason}")
    print("=" * 52 + "\n")


print("✅ llm_as_judge ready")

✅ llm_as_judge ready


In [24]:
# Cell 7-D: evaluate_batch() + summarize_batch() — for 200 QA pairs
def evaluate_batch(
    records: list,
    client: OpenAI = None,
    sleep_between: float = 0.5,
    verbose: bool = False,
) -> list:
    """
    Evaluate a list of records with llm_as_judge().

    Parameters
    ----------
    records        : list of dicts with keys: movie_input, generated_output, id (optional)
    client         : shared OpenAI client
    sleep_between  : API rate-limit buffer in seconds
    verbose        : print per-record scores if True

    Returns
    -------
    list of dicts: [{id, result}, ...]
    """
    if client is None:
        client = OpenAI()

    results = []
    total = len(records)

    for i, record in enumerate(records, 1):
        record_id = record.get("id", f"record_{i:04d}")
        print(f"[{i}/{total}] evaluating {record_id}...", end=" ", flush=True)

        result = llm_as_judge(
            movie_input=record["movie_input"],
            generated_output=record["generated_output"],
            client=client,
            verbose=verbose,
        )
        avg = result.get("average", 0.0)
        err = result.get("error")
        print(f"avg={avg:.2f}" if not err else f"ERROR: {err}")

        results.append({"id": record_id, "result": result})

        if i < total:
            time.sleep(sleep_between)

    return results


def summarize_batch(batch_results: list) -> dict:
    """
    Aggregate batch evaluation results into per-dimension and overall statistics.

    Returns
    -------
    dict with keys: n, per_dimension (mean/min/max per axis), overall_mean, error_count
    """
    per_dim = {d: [] for d in DIMENSIONS}
    averages = []
    error_count = 0

    for record in batch_results:
        r = record["result"]
        if r.get("error"):
            error_count += 1
            continue
        averages.append(r["average"])
        for dim in DIMENSIONS:
            score = r["scores"].get(dim, {}).get("score")
            if score is not None:
                per_dim[dim].append(score)

    summary = {
        "n": len(batch_results),
        "per_dimension": {},
        "overall_mean": round(sum(averages) / len(averages), 2) if averages else 0.0,
        "error_count": error_count,
    }
    for dim, vals in per_dim.items():
        if vals:
            summary["per_dimension"][dim] = {
                "mean": round(sum(vals) / len(vals), 2),
                "min": min(vals),
                "max": max(vals),
            }
    return summary


print("✅ evaluate_batch and summarize_batch ready")

✅ evaluate_batch and summarize_batch ready


## Section 8: Experiment — Idea 1 (Psychological Thriller)

**Input:** Structured dict with core_premise / thematic_core / negative_constraints  
**Variants compared:** V1 (zero-shot) vs V2 (RAG + Gemini visual grounding)  
**Fixed variables:** k=5, genre_filter=[thriller, horror, mystery], Mistral-7B NF4


In [25]:
# Cell 8-A: Idea 1 input definition
movie_input = {
    "core_premise": (
        "A low-budget psychological thriller set in a rainy rural village, "
        "where the only clinic is cut off by flooding."
    ),
    "thematic_core": "Paranoia and distrust between strangers forced into prolonged isolation.",
    "negative_constraints": (
        "Do NOT invent characters, disappearances, or a villain. "
        "The flooding is setting only — not a thriller device. "
        "Do NOT use the phrase 'Isolation breeds suspicion' or any generic isolation tagline."
    ),
}
print("Input defined.")

Input defined.


In [27]:
# Cell 8-B: Run V1 and V2
v1, v2, topk, refs = run_v1_v2(
    movie_input,
    k=5,
    genre_filter=["thriller", "horror", "mystery"]
)

print("=== Retrieved movies ===")
display(topk[["title", "genre_names", "tagline", "score"]])

print("\n=== References passed to V2 ===")
print(refs)

print("\n=== V1: Zero-Shot ===")
pprint(v1)

print("\n=== V2: RAG + Visual Grounding ===")
pprint(v2)

=== Retrieved movies ===


,title,genre_names,tagline,score
0,Take Shelter,"['Thriller', 'Drama', 'Horror']",Far away from the cruel world.,0.489664
1,Hard Rain,"['Thriller', 'Crime', 'Action']",A simple plan. An instant fortune. Just add wa...,0.457851
2,Cure,"['Crime', 'Thriller', 'Horror', 'Mystery']",Madness. Terror. Murder.,0.427415
3,Regression,"['Horror', 'Mystery', 'Thriller']",Fear always finds its victim,0.415956
4,Delirium,"['Horror', 'Thriller']",It's all in your head,0.386188



=== References passed to V2 ===
- Take Shelter: tagline: "Far away from the cruel world." | overview: "Plagued by a series of apocalyptic visions, a young husband and father questions whether to shelter his family from a coming storm, or from himself."
- Hard Rain: tagline: "A simple plan. An instant fortune. Just add water." | overview: "An armored car driver tries to elude a gang of thieves while a flood ravages the countryside."
- Cure: tagline: "Madness. Terror. Murder." | overview: "A detective starts spiraling out of control when a wave of gruesome murders with seemingly similar bizarre circumstances is sweeping Tokyo."
- Regression: tagline: "Fear always finds its victim" | overview: "Minnesota, 1990. Detective Bruce Kenner investigates the case of young Angela, who accuses her father, John Gray, of an unspeakable crime. When John unexpectedly and without recollection admits guilt,"
- Delirium: tagline: "It's all in your head" | overview: "A man recently released from a mental 

In [28]:
# Cell 8-C: Evaluate V1 and V2 with LLM-as-Judge
judge_client = OpenAI()

print(">>> Evaluating V1")
result_v1 = llm_as_judge(movie_input, v1, client=judge_client)

print(">>> Evaluating V2")
result_v2 = llm_as_judge(movie_input, v2, client=judge_client)

print(f"\nV1 average : {result_v1['average']:.2f}")
print(f"V2 average : {result_v2['average']:.2f}")
print(f"Delta (V2-V1): {result_v2['average'] - result_v1['average']:+.2f}")

>>> Evaluating V1

  average: 4.80 / 5.00
  narrative_fidelity           █████  5/5
    → The assets adhere strictly to the core premise and thematic constraints without introducing disallowed elements.
  genre_alignment              █████  5/5
    → The language and tone consistently match the psychological thriller genre.
  visual_specificity           █████  5/5
    → The description includes precise visual elements such as an ominous red moon and rain-drenched scenery.
  creative_specificity         ████░  4/5
    → The phrasing is distinct and imaginative but could benefit from a more unique metaphor.
  output_format_validity       █████  5/5
    → All required sections are within their respective word limits.

>>> Evaluating V2

  average: 4.80 / 5.00
  narrative_fidelity           █████  5/5
    → The output adheres strictly to the premise and respects all negative constraints.
  genre_alignment              █████  5/5
    → The language effectively matches the psychological thr

In [29]:
# Cell 8-D: Per-dimension comparison table
rows = []
for dim in DIMENSIONS:
    s1 = result_v1["scores"].get(dim, {}).get("score", None)
    s2 = result_v2["scores"].get(dim, {}).get("score", None)
    rows.append({
        "dimension": dim,
        "V1 (zero-shot)": s1,
        "V2 (RAG+visual)": s2,
        "delta": (s2 - s1) if (s1 is not None and s2 is not None) else None,
    })

comparison_df = pd.DataFrame(rows)
display(comparison_df)

,dimension,V1 (zero-shot),V2 (RAG+visual),delta
0,narrative_fidelity,5,5,0
1,genre_alignment,5,5,0
2,visual_specificity,5,5,0
3,creative_specificity,4,4,0
4,output_format_validity,5,5,0


## Section 9: QA Pair Generation & Validation

Cached artifacts live under **`to_be_cached/`** inside the cloned repo (same directory as `tmdb-fetch/`). Commit that folder after a full Colab run to skip expensive API calls next time. Delete a cached file to force regeneration.

| Cell | Description |
|------|-------------|
| 9-A | Generate 200 QA pairs (GPT-4o) → `to_be_cached/qa_pairs_raw.json` |
| 9-B | Mistral-7B critic → `to_be_cached/qa_pairs_validated.json` |
| 9-C | 80/20 split → `to_be_cached/qa_val.json`, `qa_test.json` |
| 9-D | Tuning grid (GPT judge) → `to_be_cached/tuning_results.csv` |

In [ ]:
# Cell 9-A: QA pair generation with GPT-4o (reverse generation from TMDB metadata)

TOP_GENRES = ["Drama", "Action", "Thriller", "Comedy", "Horror"]


def _primary_top_genre(genre_names):
    """First matching label from TOP_GENRES (fixed order) for stratification."""
    if pd.isna(genre_names):
        return None
    s = str(genre_names).strip()
    # TMDB CSV stores genre_names as a Python-list string, e.g. "['Action', 'Thriller']"
    if s.startswith("[") and s.endswith("]"):
        gset_lower = {m.group(1).strip().lower() for m in re.finditer(r"['\"]([^'\"]*)['\"]", s)}
    else:
        # Comma-separated fallback
        gset_lower = {x.strip().lower() for x in s.split(",") if x.strip()}
    for g in TOP_GENRES:
        if g.lower() in gset_lower:
            return g
    return None


def _stratified_sample_movies(df: pd.DataFrame, strat_col: str, n: int, random_state: int = 42) -> pd.DataFrame:
    """Sample n rows stratified by strat_col using a multinomial allocation (no sklearn)."""
    rng = np.random.RandomState(random_state)
    work = df[df[strat_col].notna()].copy()
    vc = work[strat_col].value_counts().sort_index()
    if vc.empty:
        raise ValueError("No rows with stratification labels")
    props = (vc / vc.sum()).values
    draws = rng.multinomial(n, props)
    parts = []
    for label, draw in zip(vc.index, draws):
        sub = work[work[strat_col] == label]
        use = min(int(draw), len(sub))
        if use > 0:
            parts.append(sub.sample(n=use, random_state=rng.randint(0, 2**31 - 1)))
    result = pd.concat(parts) if parts else work.iloc[:0]
    if len(result) < n:
        avail = work.loc[~work.index.isin(result.index)]
        need = n - len(result)
        if len(avail) >= need:
            extra = avail.sample(n=need, random_state=rng.randint(0, 2**31 - 1))
            result = pd.concat([result, extra])
    if len(result) > n:
        result = result.sample(n=n, random_state=random_state)
    return result.reset_index(drop=True)


# API outputs are stored under to_be_cached/ — if present, skip GPT-4o calls.
CACHE_DIR = os.path.join(REPO_ROOT, "to_be_cached")
os.makedirs(CACHE_DIR, exist_ok=True)
QA_RAW_CACHE = os.path.join(CACHE_DIR, "qa_pairs_raw.json")

if os.path.isfile(QA_RAW_CACHE):
    with open(QA_RAW_CACHE, "r", encoding="utf-8") as f:
        qa_pairs_raw = json.load(f)
    print(f"Loaded {len(qa_pairs_raw)} QA pairs from cache (skipped GPT-4o): {QA_RAW_CACHE}")
else:
    qa_gen_client = OpenAI()

    sample_df = corpus_df.copy()
    sample_df["_stratum"] = sample_df["genre_names"].map(_primary_top_genre)
    sample_200 = _stratified_sample_movies(sample_df, "_stratum", 200, random_state=42)

    qa_pairs_raw = []
    SYSTEM_QA = (
        "You reverse-engineer a structured creative brief from movie metadata. "
        "Output a single JSON object only. Do not copy or closely paraphrase the tagline in any field."
    )

    for _, row in tqdm(sample_200.iterrows(), total=len(sample_200), desc="GPT-4o QA generation"):
        movie_id = int(row["id"])
        title = str(row["title"])
        tagline = str(row["tagline"]) if pd.notna(row["tagline"]) else ""
        overview = str(row["overview"]) if pd.notna(row["overview"]) else ""
        genre_names = str(row["genre_names"]) if pd.notna(row["genre_names"]) else ""

        user_msg = f"""From this TMDB movie metadata, produce a JSON object with exactly these keys:
- core_premise: one sentence setting and situation (no invented plot specifics not implied below)
- thematic_core: emotional or psychological theme
- negative_constraints: explicit comma-separated list of elements NOT to invent later

Metadata:
title: {title}
tagline: {tagline}
overview: {overview}
genre_names: {genre_names}
"""

        try:
            resp = qa_gen_client.chat.completions.create(
                model="gpt-4o",
                response_format={"type": "json_object"},
                temperature=0.5,
                messages=[
                    {"role": "system", "content": SYSTEM_QA},
                    {"role": "user", "content": user_msg},
                ],
            )
            payload = json.loads(resp.choices[0].message.content.strip())
            qa_pairs_raw.append(
                {
                    "movie_id": movie_id,
                    "title": title,
                    "genre_names": genre_names,
                    "core_premise": str(payload.get("core_premise", "")).strip(),
                    "thematic_core": str(payload.get("thematic_core", "")).strip(),
                    "negative_constraints": str(payload.get("negative_constraints", "")).strip(),
                    "gold_tagline": tagline,
                    "gold_overview": overview,
                }
            )
        except Exception as e:
            print(f"[skip movie_id={movie_id}] {e}")

        time.sleep(0.3)

    with open(QA_RAW_CACHE, "w", encoding="utf-8") as f:
        json.dump(qa_pairs_raw, f, ensure_ascii=False, indent=2)

    print(f"Wrote {len(qa_pairs_raw)} records to {QA_RAW_CACHE}")

In [ ]:
# Cell 9-B: LLM critic validation with Mistral-7B

CACHE_DIR = os.path.join(REPO_ROOT, "to_be_cached")
os.makedirs(CACHE_DIR, exist_ok=True)
QA_RAW_CACHE = os.path.join(CACHE_DIR, "qa_pairs_raw.json")
QA_VALIDATED_CACHE = os.path.join(CACHE_DIR, "qa_pairs_validated.json")
LEGACY_RAW = os.path.join(REPO_ROOT, "qa_pairs_raw.json")


def _extract_critic_json(text: str) -> dict:
    """Brace-scan JSON extraction (same idea as extract_best_json) for critic schema."""
    decoder = json.JSONDecoder()
    i = 0
    best = None
    while True:
        start = text.find("{", i)
        if start == -1:
            break
        try:
            obj, end = decoder.raw_decode(text[start:])
            if isinstance(obj, dict) and "premise_faithful" in obj and "constraints_specific" in obj:
                best = obj
            i = start + end
        except json.JSONDecodeError:
            i = start + 1
    if best is None:
        raise ValueError("No critic JSON object found in model output.")
    return best


def _critic_prompt(rec: dict) -> str:
    return f"""You are a strict QA validator for reverse-generated movie briefs.

Movie ID: {rec["movie_id"]}
Title: {rec["title"]}
Original overview (ground truth for what the movie is about):
{rec["gold_overview"]}

Generated fields to check:
core_premise: {rec["core_premise"]}
thematic_core: {rec["thematic_core"]}
negative_constraints: {rec["negative_constraints"]}

Tasks:
1) Does core_premise faithfully reflect the movie overview without inventing characters, events, or settings not supported by the overview?
2) Does negative_constraints list at least two concrete elements that are clearly absent from core_premise (not vague words like "nothing")?

Reply with ONE JSON object only, keys:
"movie_id" (int), "premise_faithful" (bool), "constraints_specific" (bool), "reason" (one short sentence).
No markdown, no text outside JSON.
"""


if os.path.isfile(QA_VALIDATED_CACHE):
    with open(QA_VALIDATED_CACHE, "r", encoding="utf-8") as f:
        validated = json.load(f)
    print(f"Loaded {len(validated)} validated QA pairs from cache (skipped Mistral): {QA_VALIDATED_CACHE}")
else:
    raw_path = QA_RAW_CACHE if os.path.isfile(QA_RAW_CACHE) else LEGACY_RAW
    if not os.path.isfile(raw_path):
        raise FileNotFoundError(f"Need {QA_RAW_CACHE} (run 9-A) or legacy {LEGACY_RAW}")
    with open(raw_path, "r", encoding="utf-8") as f:
        qa_raw = json.load(f)

    validated = []
    failed = 0

    for rec in tqdm(qa_raw, desc="Mistral critic"):
        prompt = _critic_prompt(rec)
        try:
            raw_out = generate_text(prompt, max_new_tokens=256)
            parsed = _extract_critic_json(raw_out)
            pf = bool(parsed.get("premise_faithful", False))
            cs = bool(parsed.get("constraints_specific", False))
            reason = str(parsed.get("reason", "")).strip()
            if pf and cs:
                out = dict(rec)
                out["critic_reason"] = reason
                validated.append(out)
            else:
                failed += 1
        except Exception as e:
            failed += 1
            print(f"[critic fail movie_id={rec.get('movie_id')}] {e}")

    print(f"Passed: {len(validated)}  Failed: {failed}")

    with open(QA_VALIDATED_CACHE, "w", encoding="utf-8") as f:
        json.dump(validated, f, ensure_ascii=False, indent=2)

    print(f"Wrote {len(validated)} validated records to {QA_VALIDATED_CACHE}")

In [ ]:
# Cell 9-C: val / test split (80/20, stratified by primary top-5 genre)

CACHE_DIR = os.path.join(REPO_ROOT, "to_be_cached")
os.makedirs(CACHE_DIR, exist_ok=True)
QA_VALIDATED_CACHE = os.path.join(CACHE_DIR, "qa_pairs_validated.json")
QA_VAL_CACHE = os.path.join(CACHE_DIR, "qa_val.json")
QA_TEST_CACHE = os.path.join(CACHE_DIR, "qa_test.json")
LEGACY_VALIDATED = os.path.join(REPO_ROOT, "qa_pairs_validated.json")


def _stratified_train_test(df: pd.DataFrame, strat_col: str, test_size: float = 0.2, random_state: int = 42):
    rng = np.random.RandomState(random_state)
    train_idx = []
    test_idx = []
    for _, sub in df.groupby(strat_col):
        idx = sub.index.to_numpy()
        rng.shuffle(idx)
        n = len(idx)
        n_test = int(round(n * test_size))
        if n <= 1:
            train_idx.extend(idx.tolist())
        else:
            n_test = max(1, min(n_test, n - 1))
            test_idx.extend(idx[:n_test].tolist())
            train_idx.extend(idx[n_test:].tolist())
    return df.loc[train_idx], df.loc[test_idx]


if os.path.isfile(QA_VAL_CACHE) and os.path.isfile(QA_TEST_CACHE):
    with open(QA_VAL_CACHE, "r", encoding="utf-8") as f:
        qa_val_records = json.load(f)
    with open(QA_TEST_CACHE, "r", encoding="utf-8") as f:
        qa_test_records = json.load(f)
    print(f"Loaded val/test from cache (skipped split): {QA_VAL_CACHE}, {QA_TEST_CACHE}")
    qa_val_df = pd.DataFrame(qa_val_records)
    qa_test_df = pd.DataFrame(qa_test_records)
    qa_val_df["_stratum"] = qa_val_df["genre_names"].map(_primary_top_genre)
    qa_test_df["_stratum"] = qa_test_df["genre_names"].map(_primary_top_genre)
else:
    val_in = QA_VALIDATED_CACHE if os.path.isfile(QA_VALIDATED_CACHE) else LEGACY_VALIDATED
    if not os.path.isfile(val_in):
        raise FileNotFoundError(f"Need {QA_VALIDATED_CACHE} (run 9-B) or legacy {LEGACY_VALIDATED}")
    with open(val_in, "r", encoding="utf-8") as f:
        qa_validated = json.load(f)

    qa_df = pd.DataFrame(qa_validated)
    qa_df["_stratum"] = qa_df["genre_names"].map(_primary_top_genre)
    qa_strat = qa_df[qa_df["_stratum"].notna()].copy()
    if len(qa_strat) < len(qa_df):
        print(f"Dropped {len(qa_df) - len(qa_strat)} rows without a top-5 primary genre for stratification")

    qa_val_df, qa_test_df = _stratified_train_test(qa_strat, "_stratum", test_size=0.2, random_state=42)

    qa_val_records = qa_val_df.drop(columns=["_stratum"], errors="ignore").to_dict(orient="records")
    qa_test_records = qa_test_df.drop(columns=["_stratum"], errors="ignore").to_dict(orient="records")

    with open(QA_VAL_CACHE, "w", encoding="utf-8") as f:
        json.dump(qa_val_records, f, ensure_ascii=False, indent=2)
    with open(QA_TEST_CACHE, "w", encoding="utf-8") as f:
        json.dump(qa_test_records, f, ensure_ascii=False, indent=2)

print(f"Val size: {len(qa_val_records)}  Test size: {len(qa_test_records)}")
print("Val genre distribution (primary top-5):")
print(qa_val_df["_stratum"].value_counts())
print("Test genre distribution (primary top-5):")
print(qa_test_df["_stratum"].value_counts())

In [ ]:
# Cell 9-D: Hyperparameter tuning on val subset (llm_as_judge on V2 output)

CACHE_DIR = os.path.join(REPO_ROOT, "to_be_cached")
os.makedirs(CACHE_DIR, exist_ok=True)
QA_VAL_CACHE = os.path.join(CACHE_DIR, "qa_val.json")
TUNING_CACHE = os.path.join(CACHE_DIR, "tuning_results.csv")
LEGACY_QA_VAL = os.path.join(REPO_ROOT, "qa_val.json")

if os.path.isfile(TUNING_CACHE):
    tuning_df = pd.read_csv(TUNING_CACHE)
    print(f"Loaded tuning results from cache (skipped judge grid): {TUNING_CACHE}")
else:
    val_path = QA_VAL_CACHE if os.path.isfile(QA_VAL_CACHE) else LEGACY_QA_VAL
    if not os.path.isfile(val_path):
        raise FileNotFoundError(f"Need {QA_VAL_CACHE} (run 9-C) or legacy {LEGACY_QA_VAL}")
    with open(val_path, "r", encoding="utf-8") as f:
        qa_val_list = json.load(f)

    qa_val_tune = pd.DataFrame(qa_val_list).sample(n=min(20, len(qa_val_list)), random_state=42)
    tune_records = qa_val_tune.to_dict(orient="records")

    judge_client_tune = OpenAI()

    grid_k = [3, 5, 7]
    grid_gf = [True, False]
    grid_temp = [0.4, 0.6, 0.8]

    rows = []

    for k in grid_k:
        for use_gf in grid_gf:
            for temperature in grid_temp:
                scores = []
                n_ok = 0
                for rec in tune_records:
                    movie_input = {
                        "core_premise": rec["core_premise"],
                        "thematic_core": rec["thematic_core"],
                        "negative_constraints": rec["negative_constraints"],
                    }
                    if use_gf:
                        gn = str(rec.get("genre_names", "") or "")
                        gf = [x.strip().lower() for x in gn.split(",") if x.strip()]
                        genre_filter = gf if gf else None
                    else:
                        genre_filter = None
                    try:
                        _v1, v2, _topk, _refs = run_v1_v2(
                            movie_input, k=k, genre_filter=genre_filter, temperature=temperature
                        )
                        judge_out = llm_as_judge(
                            movie_input, v2, client=judge_client_tune, verbose=False
                        )
                        if judge_out.get("error"):
                            continue
                        scores.append(float(judge_out["average"]))
                        n_ok += 1
                    except Exception as e:
                        print(f"[tune skip movie_id={rec.get('movie_id')}] {e}")
                avg_score = float(np.mean(scores)) if scores else float("nan")
                rows.append(
                    {
                        "k": k,
                        "genre_filter": use_gf,
                        "temperature": temperature,
                        "avg_score": avg_score,
                        "n_evaluated": n_ok,
                    }
                )

    tuning_df = pd.DataFrame(rows)
    tuning_df.to_csv(TUNING_CACHE, index=False)
    print(f"Wrote {TUNING_CACHE}")

print("Top 3 configs by avg_score:")
print(
    tuning_df.sort_values("avg_score", ascending=False, na_position="last")
    .head(3)
    .to_string(index=False)
)